## 4 — GLMERStan (state-level estimates)
Bayesian Multilevel Regression + Poststratification via `rstanarm` (R) called from Python using `rpy2`.
Outcome: `happening_bin` (Do you think global warming is happening? Yes=1 / No=0)

**Howe (2015) 3-level architecture** estimated via full MCMC (NUTS/HMC) through `rstanarm::stan_glmer`:
- Individual: `logit(p_i) = γ₀ + α_gender + α_race + α_educ + α_state`
- State: `α_state[s] ~ N(α_region[div[s]] + γ_carbon·co2_std[s] + γ_pres·pres_std[s] + γ_drive·drive_std[s] + γ_ss·samesex_std[s], σ_state²)`
- Region: `α_region[r] ~ N(0, σ_region²)` — 9 Census divisions

Output: `outputs/estimates/glmerstan_state_estimates.csv`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()

DATA_DIR = '../test_data/processed/'
SEED = 42

In [ ]:
%load_ext rpy2.ipython

### 1. Load data

In [ ]:
survey_raw = pd.read_csv(DATA_DIR + 'climate_survey_responses_recoded.csv',
                        dtype={'state_fips': str})
ps_frame   = pd.read_csv(DATA_DIR + 'poststrat_state.csv',
                        dtype={'state_fips': str})

print(f'Survey raw:   {survey_raw.shape}')
print(f'Poststrat:    {ps_frame.shape} | {ps_frame["state_fips"].nunique()} states')

### 2. Prep survey data
 Drop Don't-Know responses for `happening_bin`

In [ ]:
DEMOG = ['gender', 'race4', 'educ_category', 'state_fips']

survey = survey_raw.dropna(subset=['happening_bin']).copy()
survey['happening_bin'] = survey['happening_bin'].astype(int)

survey['educ_category'] = survey['educ_category'].astype(str)
ps_frame['educ_category'] = ps_frame['educ_category'].astype(str)

print(f'Respondents after dropping DK: {len(survey):,}  '
      f'({survey["happening_bin"].mean()*100:.1f}% Yes)')
print()
print('Level counts in survey:')
for c in DEMOG:
    print(f'  {c}: {sorted(survey[c].unique().tolist())}')

In [ ]:
# ── State-level covariates (Howe 2015: carbon + presidential vote + ACS extras) ─
county_raw = pd.read_csv(DATA_DIR + 'poststrat_county.csv', dtype={'state_fips': str})

state_cov = (
    county_raw.groupby('state_fips')
    .apply(lambda g: pd.Series({
        'co2_per_capita':      np.average(g['co2_per_capita'],      weights=g['N_rounded']),
        'dem_share_two_party': np.average(g['dem_share_two_party'], weights=g['N_rounded']),
    }), include_groups=False)
    .reset_index()
)
state_cov['co2_per_capita'].fillna(state_cov['co2_per_capita'].mean(), inplace=True)
state_cov['dem_share_two_party'].fillna(state_cov['dem_share_two_party'].mean(), inplace=True)

acs_extra = pd.read_csv(DATA_DIR + 'acs_state_extra_covariates.csv',
                        dtype={'state_fips': str})
state_cov = state_cov.merge(acs_extra, on='state_fips')

state_div = ps_frame.groupby('state_fips')['division'].first().reset_index()
state_cov = (state_cov.merge(state_div, on='state_fips')
                       .sort_values('state_fips').reset_index(drop=True))

print(f'State covariates: {state_cov.shape} — columns: {state_cov.columns.tolist()}')
print(state_cov[['state_fips','co2_per_capita','dem_share_two_party',
                  'pct_drive_alone','pct_samesex_hh','division']].head(8).to_string(index=False))

### 3. Prep data for R
Standardise state-level covariates (z-score) and merge into survey and poststrat frames.

In [ ]:
def _std(x):
    return (x - x.mean()) / x.std()

state_cov['co2_std']     = _std(state_cov['co2_per_capita'])
state_cov['pres_std']    = _std(state_cov['dem_share_two_party'])
state_cov['drive_std']   = _std(state_cov['pct_drive_alone'])
state_cov['samesex_std'] = _std(state_cov['pct_samesex_hh'])

cov_std = ['state_fips', 'co2_std', 'pres_std', 'drive_std', 'samesex_std']

# survey doesn't have division — merge it along with standardised covariates
survey_r = survey.merge(state_cov[cov_std + ['division']], on='state_fips', how='left')
survey_r['division'] = survey_r['division'].astype(str)

# ps_frame already has division; just add standardised covariates
ps_r = ps_frame.merge(state_cov[cov_std], on='state_fips', how='left')
ps_r['division'] = ps_r['division'].astype(str)

print(f'survey_r : {survey_r.shape}')
print(f'ps_r     : {ps_r.shape}')

### 4. Fit Bayesian model via rstanarm (NUTS/HMC)

**Individual level:** `logit(p_i) = γ₀ + α_gender[g] + α_race[r] + α_educ[e] + α_state[s]`

**State level (informative prior with 4 state-level covariates):**
`α_state[s] ~ N(α_region[div[s]] + γ_carbon·co2 + γ_pres·pres + γ_drive·drive + γ_samesex·samesex, σ_state²)`

**Region level** (9 Census divisions): `α_region[r] ~ N(0, σ_region²)`

All 4 covariates standardised (z-score). Priors match Howe 2015: Intercept ~ N(0, 1.5²), γ ~ N(0, 1), σ ~ HalfNormal(2.5).

In [ ]:
%%R -i survey_r

suppressPackageStartupMessages(library(rstanarm))

survey_r$gender        <- as.factor(survey_r$gender)
survey_r$race4         <- as.factor(survey_r$race4)
survey_r$educ_category <- as.factor(survey_r$educ_category)
survey_r$state_fips    <- as.factor(survey_r$state_fips)
survey_r$division      <- as.factor(survey_r$division)

fit <- stan_glmer(
  happening_bin ~ co2_std + pres_std + drive_std + samesex_std +
    (1 | division) + (1 | state_fips) +
    (1 | gender) + (1 | race4) + (1 | educ_category),
  data             = survey_r,
  family           = binomial(link = 'logit'),
  prior            = normal(0, 1, autoscale = FALSE),
  prior_intercept  = normal(0, 1.5, autoscale = FALSE),
  prior_covariance = decov(regularization = 1, concentration = 1, shape = 1, scale = 2.5),
  chains           = 4L,
  iter             = 2000L,
  warmup           = 1000L,
  seed             = 42L,
  cores            = 4L,
  adapt_delta      = 0.9
)

cat('\n=== Fixed Effects ===\n')
print(round(fixef(fit), 4))
cat('\n=== Random Effect SDs ===\n')
print(VarCorr(fit))
cat('\n=== Convergence (Rhat, target < 1.05) ===\n')
rhat_vals <- summary(fit)[, 'Rhat']
cat(sprintf('Max Rhat: %.3f\n', max(rhat_vals, na.rm = TRUE)))

### 5. Fitted random effects (posterior means)

In [ ]:
%%R -o ranef_state_r -o ranef_gender_r -o ranef_race_r -o ranef_educ_r -o ranef_div_r

re <- ranef(fit)

ranef_state_r  <- data.frame(group = rownames(re$state_fips),
                              alpha = re$state_fips[['(Intercept)']])
ranef_gender_r <- data.frame(group = rownames(re$gender),
                              alpha = re$gender[['(Intercept)']])
ranef_race_r   <- data.frame(group = rownames(re$race4),
                              alpha = re$race4[['(Intercept)']])
ranef_educ_r   <- data.frame(group = rownames(re$educ_category),
                              alpha = re$educ_category[['(Intercept)']])
ranef_div_r    <- data.frame(group = rownames(re$division),
                              alpha = re$division[['(Intercept)']])

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
datasets = [
    (ranef_gender_r, 'gender'),
    (ranef_race_r,   'race4'),
    (ranef_educ_r,   'educ_category'),
    (ranef_div_r,    'region (Census div)'),
    (ranef_state_r.head(10), 'state (first 10)'),
]
for ax, (df, label) in zip(axes, datasets):
    ax.barh(df['group'].astype(str), df['alpha'], color='steelblue', alpha=0.8)
    ax.axvline(0, color='red', linewidth=0.8, linestyle='--')
    ax.set_title(label)
    ax.set_xlabel('posterior mean RE')
plt.tight_layout()
plt.show()

### 6. Posterior predictive on the poststrat frame
Uses `posterior_epred()` to average over all 4,000 MCMC draws, giving calibrated cell-level predictions.

In [ ]:
%%R -i ps_r -o cell_preds_r

ps_r$gender        <- as.factor(ps_r$gender)
ps_r$race4         <- as.factor(ps_r$race4)
ps_r$educ_category <- as.factor(ps_r$educ_category)
ps_r$state_fips    <- as.factor(ps_r$state_fips)
ps_r$division      <- as.factor(ps_r$division)

cat('Running posterior_epred on', nrow(ps_r), 'poststrat cells...\n')
draws <- posterior_epred(fit, newdata = ps_r, allow_new_levels = TRUE)

cell_preds_r <- data.frame(
  state_fips     = as.character(ps_r$state_fips),
  N_rounded      = ps_r$N_rounded,
  predicted_prob = colMeans(draws)
)
cat(sprintf('Predictions: n=%d  mean=%.3f  range=[%.3f, %.3f]\n',
            nrow(cell_preds_r),
            mean(cell_preds_r$predicted_prob),
            min(cell_preds_r$predicted_prob),
            max(cell_preds_r$predicted_prob)))

In [ ]:
ps_frame['predicted_prob'] = cell_preds_r['predicted_prob'].values

print(f'Predicted probabilities: '
      f'min={ps_frame["predicted_prob"].min():.3f}  '
      f'mean={ps_frame["predicted_prob"].mean():.3f}  '
      f'max={ps_frame["predicted_prob"].max():.3f}')
ps_frame[['state_fips','gender','race4','educ_category',
          'N_rounded','predicted_prob']].head(8)

### 7. Poststratification
Weighted average of cell-level predicted probabilities, weighted by ACS population counts.

In [ ]:
result = (
    ps_frame
    .groupby('state_fips')
    .apply(lambda g: np.average(g['predicted_prob'], weights=g['N_rounded']),
           include_groups=False)
    .reset_index(name='happening_estimate')
)

STATE_NAMES = {
    '01':'Alabama','02':'Alaska','04':'Arizona','05':'Arkansas','06':'California',
    '08':'Colorado','09':'Connecticut','10':'Delaware','11':'District of Columbia',
    '12':'Florida','13':'Georgia','15':'Hawaii','16':'Idaho','17':'Illinois',
    '18':'Indiana','19':'Iowa','20':'Kansas','21':'Kentucky','22':'Louisiana',
    '23':'Maine','24':'Maryland','25':'Massachusetts','26':'Michigan',
    '27':'Minnesota','28':'Mississippi','29':'Missouri','30':'Montana',
    '31':'Nebraska','32':'Nevada','33':'New Hampshire','34':'New Jersey',
    '35':'New Mexico','36':'New York','37':'North Carolina','38':'North Dakota',
    '39':'Ohio','40':'Oklahoma','41':'Oregon','42':'Pennsylvania',
    '44':'Rhode Island','45':'South Carolina','46':'South Dakota',
    '47':'Tennessee','48':'Texas','49':'Utah','50':'Vermont',
    '51':'Virginia','53':'Washington','54':'West Virginia','55':'Wisconsin',
    '56':'Wyoming',
}
result['state_name'] = result['state_fips'].map(STATE_NAMES)
result = result.sort_values('happening_estimate', ascending=False).reset_index(drop=True)

print(f'National weighted estimate: {np.average(result["happening_estimate"]):.3f}')
print()
print('Top 10 states (highest % believing GW is happening):')
print(result.head(10).to_string(index=False))
print()
print('Bottom 10 states:')
print(result.tail(10).to_string(index=False))

### 8. Visualise state estimates

In [ ]:
fig, ax = plt.subplots(figsize=(8, 12))
plot_df = result.sort_values('happening_estimate')
ax.barh(plot_df['state_name'], plot_df['happening_estimate'], color='steelblue', alpha=0.8)
ax.axvline(np.average(result['happening_estimate']), color='red',
           linestyle='--', linewidth=1.2, label='National avg')
ax.set_xlabel('P(GW is happening)', fontsize=12)
ax.set_title('GLMERStan — State-level MRP Estimates\n\'Is global warming happening?\'',
             fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 9. Save results

In [ ]:
OUT = '../outputs/estimates/glmerstan_state_estimates.csv'
result[['state_fips','state_name','happening_estimate']].rename(columns={'happening_estimate':'estimate'}).to_csv(OUT, index=False)
print(f'Saved -> {OUT}')
print(result[['state_fips','state_name','happening_estimate']].to_string(index=False))